# 06 · Benchmark, reproducibility, and late-data tests

This notebook is an evaluation notebook, not part of the daily ingestion chain.

* **A — Step timings:** summarizes the timings already written by the pipeline.
* **B — CSV versus API JSON:** an optional benchmark using EPA's pre-generated bulk CSV.
  The CSV is benchmark-only; it is not required by the operational daily pipeline.
* **C — Reproducibility:** reruns the current Bronze → Silver → Gold notebooks and checks
  that analytical values do not change when the source files have not changed.
* **D — Late data:** optionally adds a temporary revised EPA file and checks that the
  affected county-day is rebuilt. It is disabled by default because it intentionally
  changes the current analytical result while the test is running.

The operational notebooks currently used by the job are:
`01_extract_eta`, `01_extract_firms`, `01_extract_noaa`, `02_load_bronze`,
`03_transform_silver`, `04_kpi_gold`, and `05_analytics_data_quality`.

In [0]:
%run ./00_config

# 00 · Config
Shared settings for every notebook. Every other notebook loads this one with `%run ./00_config`.

**Before the first run:** change `CATALOG` / `SCHEMA` if needed, then run `01_setup` once to create the schema and volume.

Config loaded: workspace.wildfire, raw files in /Volumes/workspace/wildfire/raw


## Test controls

`run_csv_benchmark=auto` runs the CSV comparison only when
`raw/epa_bulk/daily_88101_2020.csv` exists. It does not fail when the file has not
been uploaded. Set it to `true` to require the file, or `false` to skip the section.

The late-data test is opt-in. It creates a temporary file in `raw/epa`, runs the
current ingestion notebooks, reports the changed county-days, and removes the
temporary file. The test is intended for a controlled run because the resulting
revised record remains the latest source version in the Delta tables.

In [0]:
dbutils.widgets.dropdown(
    "run_csv_benchmark", "auto", ["auto", "true", "false"],
    "Run optional EPA bulk CSV benchmark"
)
dbutils.widgets.dropdown(
    "run_reproducibility_test", "true", ["true", "false"],
    "Run Bronze-Silver-Gold rerun test"
)
dbutils.widgets.dropdown(
    "run_late_data_test", "false", ["true", "false"],
    "Run temporary late-data correction test"
)
dbutils.widgets.text("late_test_day", "2020-09-10", "EPA day for late-data test")

RUN_CSV = dbutils.widgets.get("run_csv_benchmark")
RUN_REPRO = dbutils.widgets.get("run_reproducibility_test") == "true"
RUN_LATE = dbutils.widgets.get("run_late_data_test") == "true"
LATE_TEST_DAY = dbutils.widgets.get("late_test_day").strip()
REPS = 3
NOTEBOOK_TIMEOUT_SECONDS = 3600

# These names match the notebooks currently used by the job. Do not rename the files
# just to make this notebook work.
BRONZE_NOTEBOOK = "./02_load_bronze"
SILVER_NOTEBOOK = "./03_transform_silver"
GOLD_NOTEBOOK = "./04_kpi_gold"

print("Benchmark controls:", {
    "run_csv_benchmark": RUN_CSV,
    "run_reproducibility_test": RUN_REPRO,
    "run_late_data_test": RUN_LATE,
    "late_test_day": LATE_TEST_DAY,
})

Benchmark controls: {'run_csv_benchmark': 'auto', 'run_reproducibility_test': True, 'run_late_data_test': False, 'late_test_day': '2020-09-10'}


## A · Pipeline step timings

In [0]:
if spark.catalog.tableExists(t("ops_step_log")):
    display(spark.sql(f"""
        SELECT layer, step,
               count(*) AS runs,
               sum(rows) AS rows,
               round(avg(seconds), 1) AS avg_s,
               round(percentile(seconds, 0.5), 1) AS median_s,
               round(max(seconds), 1) AS max_s
        FROM {t('ops_step_log')}
        GROUP BY layer, step
        ORDER BY avg_s DESC
    """))
else:
    print("ops_step_log does not exist yet; run the pipeline first.")

layer,step,runs,rows,avg_s,median_s,max_s
bronze,bronze_firms,16,207707,29.2,30.1,83.9
bronze,bronze_epa,16,93555,28.6,30.6,77.3
gold,gold_fact_county_day,18,50286,22.0,21.7,32.9
bronze,bronze_noaa,16,138154,19.7,27.1,34.6
silver,silver_noaa,16,379032,17.8,18.9,27.9
silver,silver_firms,16,537028,16.2,15.3,33.2
silver,silver_epa,16,45918,13.3,12.9,18.7


## B · Optional EPA bulk CSV versus API JSON

This section preserves the proposal's CSV-versus-API comparison without making a
bulk load part of the daily job. The CSV is filtered to California and the 2020 study
window so it can be compared with the API JSON files already in `raw/epa`.

If the CSV is not present, `bench_epa_from_csv` is deliberately **not** created.
`bench_epa_from_json` can still be created from the operational API output.

In [0]:
import os
import json as _json
import re
import statistics
import uuid

bulk_path = f"{LANDING['epa_csv_bulk']}/daily_88101_2020.csv"
json_glob = f"{LANDING['epa']}/*.json"

def path_exists(path):
    try:
        return any(f.name == os.path.basename(path) for f in dbutils.fs.ls(os.path.dirname(path)))
    except Exception as exc:
        print(f"Could not inspect {path}: {exc}")
        return False

bulk_available = path_exists(bulk_path)
if RUN_CSV == "true" and not bulk_available:
    raise FileNotFoundError(
        f"Required benchmark input is missing: {bulk_path}. "
        "Upload daily_88101_2020.csv or set run_csv_benchmark=false/auto."
    )

run_csv_section = RUN_CSV == "true" or (RUN_CSV == "auto" and bulk_available)
if not run_csv_section:
    print(
        "CSV benchmark skipped: daily_88101_2020.csv is not present in "
        f"{LANDING['epa_csv_bulk']}. No bench_epa_from_csv table was created."
    )

def api_epa_df():
    """Return API rows in the same California/study-window scope as the CSV test."""
    return (
        spark.read.option("multiLine", True).json(json_glob)
        .select(F.explode("Data").alias("r"))
        .select("r.*")
        .where(
            (F.lpad(F.col("state_code").cast("string"), 2, "0") == STATE_FIPS)
            & F.to_date("date_local").between(F.lit(START_DATE), F.lit(END_DATE))
            & (F.col("parameter_code").cast("string") == F.lit("88101"))
        )
    )

def bulk_epa_df():
    """Return bulk CSV rows in the same California/study-window scope."""
    state = F.lpad(F.regexp_replace(F.col("State Code"), "[^0-9]", ""), 2, "0")
    return (
        spark.read.option("header", True).option("inferSchema", False).csv(bulk_path)
        .where(
            (state == STATE_FIPS)
            & F.to_date("Date Local").between(F.lit(START_DATE), F.lit(END_DATE))
            & (F.col("Parameter Code").cast("string") == F.lit("88101"))
        )
    )

def timed_count(df, value_col):
    started = time.time()
    row = df.agg(
        F.count("*").alias("rows"),
        F.avg(F.col(value_col).cast("double")).alias("avg_value"),
    ).first()
    return time.time() - started, int(row["rows"] or 0)

benchmark_rows = []

def add_timing_result(test, status, times, rows, note=""):
    benchmark_rows.append((
        test,
        status,
        int(rows),
        float(statistics.median(times)) if times else None,
        float(min(times)) if times else None,
        float(max(times)) if times else None,
        note,
    ))

if run_csv_section:
    for test, make_df, value_col in [
        ("raw_csv_bulk", bulk_epa_df, "Arithmetic Mean"),
        ("raw_json_api", api_epa_df, "arithmetic_mean"),
    ]:
        times = []
        rows = 0
        for _ in range(REPS):
            elapsed, rows = timed_count(make_df(), value_col)
            times.append(elapsed)
        add_timing_result(test, "completed", times, rows)

    # Materialise the two scoped inputs as Delta tables for a second comparison.
    # The CSV table exists only when the optional file exists.
    bulk_epa_df().write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        t("bench_epa_from_csv")
    )
    api_epa_df().write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        t("bench_epa_from_json")
    )

    delta_queries = [
        (
            "delta_query_from_csv",
            f"""SELECT `County Code`, avg(CAST(`Arithmetic Mean` AS DOUBLE)) AS avg_pm25
                FROM {t('bench_epa_from_csv')} GROUP BY 1""",
        ),
        (
            "delta_query_from_json",
            f"""SELECT county_code, avg(CAST(arithmetic_mean AS DOUBLE)) AS avg_pm25
                FROM {t('bench_epa_from_json')} GROUP BY 1""",
        ),
    ]
    for test, sql in delta_queries:
        times = []
        rows = 0
        for _ in range(REPS):
            started = time.time()
            rows = spark.sql(sql).count()
            times.append(time.time() - started)
        add_timing_result(test, "completed", times, rows)
else:
    add_timing_result(
        "raw_csv_bulk", "skipped_missing_input", [], 0,
        f"Upload {bulk_path} for the proposal's CSV-versus-API comparison.",
    )
    # The API side is still useful and uses only operationally landed data.
    times = []
    rows = 0
    for _ in range(REPS):
        elapsed, rows = timed_count(api_epa_df(), "arithmetic_mean")
        times.append(elapsed)
    add_timing_result("raw_json_api", "completed", times, rows)
    api_epa_df().write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
        t("bench_epa_from_json")
    )

if benchmark_rows:
    bench_schema = """
        test string,
        status string,
        rows long,
        median_s double,
        min_s double,
        max_s double,
        note string
    """
    bench = (
        spark.createDataFrame(benchmark_rows, bench_schema)
        .withColumn("run_at_utc", F.current_timestamp())
    )
    (
        bench.write.format("delta").mode("append")
        .option("mergeSchema", "true")
        .saveAsTable(t("bench_results"))
    )
    display(bench)

try:
    bulk_files = dbutils.fs.ls(LANDING["epa_csv_bulk"])
    for f in bulk_files:
        print(f"bulk: {f.name} {round(f.size / 1e6, 1)} MB")
except Exception:
    pass
try:
    json_files = dbutils.fs.ls(LANDING["epa"])
    print("API JSON total MB:", round(sum(f.size for f in json_files) / 1e6, 1))
except Exception:
    pass

CSV benchmark skipped: daily_88101_2020.csv is not present in /Volumes/workspace/wildfire/raw/epa_bulk. No bench_epa_from_csv table was created.


test,status,rows,median_s,min_s,max_s,note,run_at_utc
raw_csv_bulk,skipped_missing_input,0,null,null,null,Upload /Volumes/workspace/wildfire/raw/epa_bulk/daily_88101_2020.csv for the proposal's CSV-versus-API comparison.,2026-11-01T06:50:48.058Z
raw_json_api,completed,51095,4.308850049972534,3.706477642059326,6.020965576171875,,2026-11-01T06:50:48.058Z


API JSON total MB: 78.5


## C · Reproducibility: rerun unchanged inputs

This uses the current notebook names. Gold is forced to rebuild every study date,
which makes the test stricter than a normal incremental run. The fingerprint excludes
ingestion/build timestamps and compares the analytical values only.

In [0]:
def fingerprint(table):
    df = spark.table(t(table))
    ignored = {"_gold_built_at_utc", "_ingested_at_utc", "_batch_id", "_source_file"}
    cols = [c for c in df.columns if c not in ignored]
    if not cols:
        return df.count(), None
    return df.count(), df.select(
        F.sum(F.xxhash64(*cols).cast("decimal(38,0)")).alias("fingerprint")
    ).first()["fingerprint"]

def run_current_pipeline(rebuild_all=False):
    """Run the current Bronze → Silver → Gold notebooks in dependency order."""
    dbutils.notebook.run(BRONZE_NOTEBOOK, NOTEBOOK_TIMEOUT_SECONDS)
    dbutils.notebook.run(SILVER_NOTEBOOK, NOTEBOOK_TIMEOUT_SECONDS)
    params = {"rebuild_all": "true"} if rebuild_all else {}
    dbutils.notebook.run(GOLD_NOTEBOOK, NOTEBOOK_TIMEOUT_SECONDS, params)

if RUN_REPRO:
    fingerprint_tables = [
        "silver_fire_detections",
        "silver_pm25_daily",
        "silver_weather_obs",
        "gold_fact_county_day",
    ]
    before = {tb: fingerprint(tb) for tb in fingerprint_tables}
    gold_version_before = spark.sql(
        f"DESCRIBE HISTORY {t('gold_fact_county_day')} LIMIT 1"
    ).first().version

    run_current_pipeline(rebuild_all=True)

    after = {tb: fingerprint(tb) for tb in fingerprint_tables}
    repro_rows = [
        (tb, before[tb][0], after[tb][0], before[tb][1], after[tb][1], before[tb] == after[tb])
        for tb in fingerprint_tables
    ]
    repro_schema = "table string, rows_before long, rows_after long, fingerprint_before decimal(38,0), fingerprint_after decimal(38,0), unchanged boolean"
    display(spark.createDataFrame(repro_rows, repro_schema))
    print("All fingerprints unchanged:", all(r[-1] for r in repro_rows))
    display(
        spark.sql(f"DESCRIBE HISTORY {t('gold_fact_county_day')}")
        .where(F.col("version") > F.lit(gold_version_before))
        .select("version", "operation", "operationMetrics")
    )
else:
    print("Reproducibility test skipped by widget.")

table,rows_before,rows_after,fingerprint_before,fingerprint_after,unchanged
silver_fire_detections,207707,207707,-1155511944893039386361,-1155511944893039386361,true
silver_pm25_daily,13280,13280,371726175241025434172,371726175241025434172,true
silver_weather_obs,137036,137036,3001849846173031671102,3001849846173031671102,true
gold_fact_county_day,3712,3712,102570733083674567422,102570733083674567422,true


All fingerprints unchanged: True


version,operation,operationMetrics
32,MERGE,"Map(numTargetRowsCopied -> 0, numTargetRowsDeleted -> 0, numTargetFilesAdded -> 0, numTargetBytesAdded -> 0, numTargetBytesRemoved -> 0, numTargetDeletionVectorsAdded -> 0, numTargetRowsMatchedUpdated -> 0, executionTimeMs -> 5288, materializeSourceTimeMs -> 2128, numTargetRowsInserted -> 0, numTargetRowsMatchedDeleted -> 0, numTargetDeletionVectorsUpdated -> 0, scanTimeMs -> 928, numTargetRowsUpdated -> 0, numOutputRows -> 0, numTargetDeletionVectorsRemoved -> 0, numTargetRowsNotMatchedBySourceUpdated -> 0, numTargetChangeFilesAdded -> 0, numSourceRows -> 3712, numTargetFilesRemoved -> 0, numTargetRowsNotMatchedBySourceDeleted -> 0, rewriteTimeMs -> 2200)"


## D · Late-data correction test (opt-in)

A temporary copy of one non-empty EPA JSON file is created with a changed PM2.5 value
and a later `date_of_last_change`. The file name contains the same measurement date, so
the existing incremental Bronze logic sees it as a new source version. After Bronze →
Silver → Gold, the notebook reports the county-days whose Gold PM2.5 value changed.

The temporary file is removed afterward, but the revised record remains in the Delta
tables as the latest ingested version. Run this only when that state is acceptable, or
restore the tables from Delta history afterward.

In [0]:
def select_late_test_file(day):
    candidates = [
        f for f in dbutils.fs.ls(LANDING["epa"])
        if f.name.startswith(f"epa_{day}") and f.name.endswith(".json")
    ]
    if not candidates:
        raise FileNotFoundError(f"No EPA JSON file found for {day} in {LANDING['epa']}")
    # Build a local /Volumes path instead of relying on FileInfo.path, which may
    # be returned as dbfs:/... and cannot be passed to Python open().
    return f"{LANDING['epa']}/{sorted(candidates, key=lambda x: x.name)[0].name}"

def choose_valid_epa_record(data):
    for record in data:
        try:
            value = float(record.get("arithmetic_mean"))
            completeness = float(record.get("observation_percent"))
            if (
                str(record.get("state_code")).zfill(2) == STATE_FIPS
                and str(record.get("parameter_code")) == "88101"
                and PM25_MIN <= value <= PM25_MAX
                and completeness >= 75
            ):
                return record
        except (TypeError, ValueError):
            continue
    raise ValueError("No valid EPA PM2.5 record was found for the late-data test.")

if RUN_LATE:
    original_path = select_late_test_file(LATE_TEST_DAY)
    temp_name = f"epa_{LATE_TEST_DAY}_late_test_{uuid.uuid4().hex[:8]}.json"
    temp_path = f"{LANDING['epa']}/{temp_name}"

    # Volume paths are available to the driver in the same way as the extractor and
    # Bronze notebook already use them.
    with open(original_path, "r") as fh:
        doc = _json.load(fh)
    target = choose_valid_epa_record(doc.get("Data", []))
    old_value = float(target["arithmetic_mean"])
    new_value = (
        round(old_value + 10.0, 1)
        if old_value <= PM25_MAX - 10.0
        else round(old_value - 10.0, 1)
    )
    target["arithmetic_mean"] = str(new_value)
    target["date_of_last_change"] = "2099-12-31"
    with open(temp_path, "w") as fh:
        _json.dump(doc, fh)

    target_county = str(target.get("state_code")).zfill(2) + str(target.get("county_code")).zfill(3)
    print(
        f"Late-data test file: {temp_name}; county={target_county}; "
        f"date={LATE_TEST_DAY}; PM2.5 {old_value} -> {new_value}"
    )

    try:
        before_gold = (
            spark.table(t("gold_fact_county_day"))
            .select("county_fips", "date_local", "pm25_mean_ugm3")
            .cache()
        )
        before_gold.count()
        run_current_pipeline(rebuild_all=False)
        after_gold = (
            spark.table(t("gold_fact_county_day"))
            .select("county_fips", "date_local", F.col("pm25_mean_ugm3").alias("pm25_after"))
        )
        diff = (
            before_gold.join(after_gold, ["county_fips", "date_local"])
            .where(~F.col("pm25_mean_ugm3").eqNullSafe(F.col("pm25_after")))
        )
        print("Changed county-days:", diff.count())
        display(diff.orderBy("date_local", "county_fips"))
    finally:
        try:
            os.remove(temp_path)
            print(f"Removed temporary test file: {temp_name}")
        except FileNotFoundError:
            pass
else:
    print("Late-data test skipped. Set run_late_data_test=true to run it.")


Late-data test skipped. Set run_late_data_test=true to run it.
